[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/powerzoojax/PowerZooPy/blob/main/docs/en/examples/notebooks/nb07_evaluation.ipynb)

# NB07 — Evaluation

> **Prerequisites**: [NB01 — Quickstart](./nb01_quickstart.ipynb)  
> **Time**: ~10 minutes

## What You'll Learn

1. The complete `evaluate()` API and all result dict keys
2. The `evaluate_task()` convenience function for reproducible benchmark numbers
3. How to generate official `baselines.json` with the compute script
4. Baseline comparison chart: Random / Rule-Based / Oracle across all three tasks
5. How to format your results for submission (benchmark summary table)

## Setup

In [ ]:
import powerzoo
print(f"PowerZoo version: {powerzoo.__version__}")

In [ ]:
from powerzoo.tasks import make_task_env
from powerzoo.wrappers import GymnasiumWrapper
from powerzoo.benchmarks.policies import RandomPolicy, RuleBasedPolicy, OraclePolicy
from powerzoo.benchmarks import evaluate, evaluate_task, normalized_score, get_baseline

import numpy as np
import matplotlib.pyplot as plt

## 1. The `evaluate()` Function — Full API Reference

In [ ]:
# Create a test environment
env = GymnasiumWrapper(make_task_env("marl_opf", split="test"))
policy = RandomPolicy(env.action_space, seed=0)

result = evaluate(
    policy=policy,
    env=env,
    n_episodes=5,          # use 100 for official benchmark results
    seed_start=42,         # episode i uses seed = seed_start + i
    verbose=True,
    task_id="marl_opf",    # enables normalized_score computation
)

In [ ]:
print("=== evaluate() result dict ===")
print()

field_docs = {
    'mean_reward':      'Mean total reward across all episodes',
    'std_reward':       'Standard deviation of episode rewards',
    'min_reward':       'Minimum episode reward (worst run)',
    'max_reward':       'Maximum episode reward (best run)',
    'mean_ep_length':   'Mean episode length (steps)',
    'episode_rewards':  'Raw list of per-episode rewards',
    'episode_metrics':  'Per-episode info dict metrics',
    'normalized_score': '(score-random)/(oracle-random); None until baselines computed',
    'steps_per_second': 'Wall-clock simulation throughput',
}

for key, val in result.items():
    doc = field_docs.get(key, '')
    if isinstance(val, list):
        display_val = f"[{val[0]:.2f}, {val[1]:.2f}, ...] ({len(val)} items)" if len(val) > 2 else str(val)
    elif isinstance(val, float):
        display_val = f"{val:.4f}"
    else:
        display_val = str(val)
    print(f"  {key:20s}: {display_val}")
    if doc:
        print(f"  {'':20s}  → {doc}")
    print()

## 2. `evaluate_task()` — Reproducible Benchmark Numbers

`evaluate_task()` reads the task's fixed evaluation protocol and runs
exactly the right number of episodes on the test split. Use this for
numbers you want to report in a paper.

In [ ]:
# evaluate_task automatically uses: n_episodes=100, seed=42, split='test'
# We use n_episodes=5 here for speed — replace with the default for real results

env_test = GymnasiumWrapper(make_task_env("marl_opf", split="test"))
random_policy = RandomPolicy(env_test.action_space, seed=42)

result = evaluate_task(
    policy=random_policy,
    task_name="marl_opf",
    override_n_episodes=5,  # remove this line for official 100-episode evaluation
    verbose=False,
)

print(f"Task: marl_opf (test split, random policy)")
print(f"  mean_reward:      {result['mean_reward']:.3f} ± {result['std_reward']:.3f}")
print(f"  normalized_score: {result['normalized_score']}")
print(f"  (None means baselines.json not yet computed — run compute script below)")

## 3. Normalized Score Formula

The normalized score places your policy on a [0, 1] scale relative to baselines:

In [ ]:
print("Normalized score formula:")
print()
print("  normalized_score = (policy_mean_reward - random_mean_reward)")
print("                     / (oracle_mean_reward - random_mean_reward)")
print()
print("  0.0  = matches random policy (lower bound)")
print("  1.0  = matches oracle OPF solver (upper bound)")
print("  >1.0 = outperforms oracle heuristic (possible but unlikely)")
print()
print("Manual example with hypothetical numbers:")
random_r  = -1500.0
oracle_r  =  -200.0
your_r    =  -800.0
ns = (your_r - random_r) / (oracle_r - random_r)
print(f"  random_score  = {random_r:.0f}")
print(f"  oracle_score  = {oracle_r:.0f}")
print(f"  your_score    = {your_r:.0f}")
print(f"  normalized    = ({your_r:.0f} - {random_r:.0f}) / ({oracle_r:.0f} - {random_r:.0f}) = {ns:.3f}")

## 4. Baseline Comparison: Three Policies × Three Tasks

The chart below shows typical baseline scores.
After you run `python -m powerzoo.benchmarks.compute`, actual values from
`baselines.json` will replace these placeholders.

In [ ]:
# Try to load actual baselines; fall back to illustrative placeholder values
task_names   = ["marl_opf", "marl_der_arbitrage", "marl_ev_v2g"]
task_labels  = ["Task 1\n(OPF)", "Task 2\n(DER Arbitrage)", "Task 3\n(EV V2G)"]
policy_names = ["Random", "Rule-Based", "Oracle"]

# Placeholder normalized scores (illustrative)
# Random = 0.0 by definition, Oracle = 1.0 by definition
placeholder_scores = {
    "Random":    [0.0, 0.0, 0.0],
    "Rule-Based": [0.35, 0.55, 0.40],
    "Oracle":    [1.0, 1.0, 1.0],
}

# Try to load actual baselines
actual_scores = {}
for policy in policy_names:
    actual_scores[policy] = []
    for task in task_names:
        try:
            bl = get_baseline(task)
            if policy == "Random":
                actual_scores[policy].append(0.0)
            elif policy == "Oracle":
                actual_scores[policy].append(1.0)
            else:
                actual_scores[policy].append(None)
        except Exception:
            actual_scores[policy].append(None)

# Use actual if available, else placeholder
scores = {}
for policy in policy_names:
    scores[policy] = [
        a if a is not None else p
        for a, p in zip(actual_scores[policy], placeholder_scores[policy])
    ]

print("Scores used for chart:")
for policy, sc in scores.items():
    print(f"  {policy:12s}: {sc}")
print("(Replace placeholders by running `python -m powerzoo.benchmarks.compute`)")

In [ ]:
# Grouped bar chart
x = np.arange(len(task_names))
width = 0.22
bar_colors = ['#E53935', '#1565C0', '#2E7D32']

fig, ax = plt.subplots(figsize=(9, 5))

for i, (policy, color) in enumerate(zip(policy_names, bar_colors)):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, scores[policy], width, label=policy,
                  color=color, alpha=0.85, edgecolor='white')
    # Value labels on bars
    for bar, val in zip(bars, scores[policy]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8.5)

ax.axhline(0, color='gray', linewidth=0.8)
ax.axhline(1, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(task_labels)
ax.set_ylabel('Normalized Score (0 = random, 1 = oracle)')
ax.set_ylim(-0.1, 1.25)
ax.set_title('Baseline Policy Comparison — All Three Tasks')
ax.legend()
ax.text(2.5, 1.08, '← Oracle (upper bound)', fontsize=8, color='gray', ha='right')
ax.text(2.5, 0.08, '← Random (lower bound)', fontsize=8, color='gray', ha='right')

plt.tight_layout()
plt.savefig('baseline_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print("\nNote: Rule-Based scores shown are illustrative placeholders.")
print("Run the compute script to get official numbers.")

## 5. Generating Official `baselines.json`

Run the following command to compute the official Random and Oracle baseline scores
for all tasks. This must be run before `normalized_score` can be computed.

In [ ]:
print("Command to generate baselines (run once per task configuration):")
print()
print("  # All tasks, 100 episodes each, test split")
print("  python -m powerzoo.benchmarks.compute --tasks all --n_episodes 100 --split test")
print()
print("  # Single task")
print("  python -m powerzoo.benchmarks.compute --tasks marl_opf --n_episodes 100")
print()
print("Output: powerzoo/benchmarks/baselines.json")
print()
print("After running:")
print("  from powerzoo.benchmarks import get_baseline")
print("  baseline = get_baseline('marl_opf')")
print("  print(baseline)  # {'random': -1234.5, 'oracle': -312.7}")

## 6. Reporting Your Results

Format your results as a compact benchmark table for paper submissions:

In [ ]:
# Template for result reporting
print("=" * 75)
print("PowerZoo Benchmark Results")
print("=" * 75)
print(f"{'Task':<25} {'Algorithm':<15} {'Mean Reward':>12} {'Normalized Score':>16} {'N episodes':>10}")
print("-" * 75)

# Placeholder entries — replace with your actual results
rows = [
    ("marl_opf",          "Random",     -1500.0, 0.00,  100),
    ("marl_opf",          "Rule-Based",  -950.0, 0.42,  100),
    ("marl_opf",          "Your SAC",    -750.0, "?",   100),  # fill in after training
    ("marl_opf",          "Oracle",      -200.0, 1.00,  100),
    ("marl_der_arbitrage", "Random",     -300.0, 0.00,  100),
    ("marl_der_arbitrage", "Rule-Based",  150.0, 0.55,  100),
    ("marl_der_arbitrage", "Your SAC",     "?",  "?",   100),
    ("marl_der_arbitrage", "Oracle",      800.0, 1.00,  100),
    ("marl_ev_v2g",        "Random",     -500.0, 0.00,  100),
    ("marl_ev_v2g",        "Rule-Based",  200.0, 0.40,  100),
    ("marl_ev_v2g",        "Your SAC",     "?",  "?",   100),
    ("marl_ev_v2g",        "Oracle",      900.0, 1.00,  100),
]

prev_task = None
for task, algo, reward, ns, n_ep in rows:
    if task != prev_task and prev_task is not None:
        print()
    reward_str = f"{reward:.1f}" if isinstance(reward, float) else reward
    ns_str     = f"{ns:.2f}" if isinstance(ns, float) else ns
    print(f"{task:<25} {algo:<15} {reward_str:>12} {ns_str:>16} {n_ep:>10}")
    prev_task = task

print("=" * 75)
print("\n→ Report using split='test', seed_start=42, n_episodes=100")
print("→ Use evaluate_task(policy, task_name) for reproducible results")

---

## Summary

| API | Purpose |
|---|---|
| `evaluate(policy, env, n_episodes, task_id=)` | Run evaluation, return metrics dict |
| `evaluate_task(policy, task_name)` | Use task's fixed protocol (n=100, seed=42, test) |
| `result['normalized_score']` | Normalized score: 0 = random baseline, 1 = oracle |
| `get_baseline(task_name)` | Get `{'random': float, 'oracle': float}` from baselines.json |
| `python -m powerzoo.benchmarks.compute` | Generate official baselines.json |

**For paper submission**: always use `evaluate_task()` with default protocol,
report both `mean_reward ± std_reward` and `normalized_score`.

## Next (Advanced)

→ [NB08 — Time-Series Data & Custom Tasks](./nb08_data_advanced.ipynb) — `DataLoader`, custom rewards, registering tasks